# Zalo AI Challenge 2025 - RoadBuddy
## Inference Notebook for Time Measurement

This notebook is used by the organizers to measure inference time.

**Important**: BTC will run all cells sequentially. Ensure all cells run without errors.

### Step 1: Set Seed (for reproducibility)

In [ ]:
import os
import torch
import random
import numpy as np

def seed_everything(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Set seed (competition requirement)
seed_everything(42)

print("✓ Random seed set to 42")

### Step 2: Load Model and Resources

In [ ]:
import json
import time
import re
from pathlib import Path
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
from qwen_vl_utils import process_vision_info

# Configuration
CHECKPOINT_PATH = "./saved_models/checkpoint-latest"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model from {CHECKPOINT_PATH}...")
print(f"Device: {DEVICE}\n")

# Load base model
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map=DEVICE,
    trust_remote_code=True
)
print("✓ Base model loaded")

# Load LoRA adapter
model = PeftModel.from_pretrained(model, CHECKPOINT_PATH)
print("✓ LoRA adapter loaded")

# Load merger weights
merger_path = Path(CHECKPOINT_PATH) / "merger_weights.bin"
if merger_path.exists():
    merger_weights = torch.load(merger_path, map_location=DEVICE)
    for name, param in model.named_parameters():
        if name in merger_weights:
            param.data.copy_(merger_weights[name])
    print(f"✓ Loaded {len(merger_weights)} merger parameters")

# Load processor
processor = AutoProcessor.from_pretrained(CHECKPOINT_PATH, trust_remote_code=True)
print("✓ Processor loaded")

model.eval()
print("\n✅ Model ready for inference!")

### Step 3: Load Test Cases

In [ ]:
# Load test data
INPUT_JSON = "/data/test.json"

with open(INPUT_JSON, 'r', encoding='utf-8') as f:
    test_data = json.load(f)

test_cases = test_data['data'] if 'data' in test_data else test_data

print(f"Loaded {len(test_cases)} test cases")
print(f"\nFirst test case:")
print(f"  ID: {test_cases[0]['id']}")
print(f"  Question: {test_cases[0]['question'][:50]}...")
print(f"  Choices: {len(test_cases[0]['choices'])} options")

### Step 4: Run Inference and Measure Time

In [ ]:
from tqdm.notebook import tqdm

# Helper functions
def format_prompt(question, choices):
    """Format prompt to match training format"""
    prompt = f"<video>\n{question}\n\n"
    for choice in choices:
        prompt += f"{choice}\n"
    return prompt.rstrip()

def extract_answer(response):
    """Extract A/B/C/D from model response"""
    # Priority 1: "Đáp án: X" format
    match = re.search(r'(?:Đáp án|đáp án)[:\s]+([ABCD])', response, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    # Priority 2: Standalone A/B/C/D
    match = re.search(r'\b([ABCD])\b', response.upper())
    if match:
        return match.group(1)
    # Priority 3: Starts with A/B/C/D
    if response.strip() and response.strip().upper()[0] in 'ABCD':
        return response.strip().upper()[0]
    return "A"

def predict_single(video_path, question, choices):
    """Predict answer for a single test case"""
    prompt = format_prompt(question, choices)
    
    messages = [{
        "role": "user",
        "content": [
            {"type": "video", "video": str(video_path)},
            {"type": "text", "text": prompt},
        ],
    }]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(DEVICE)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    
    generated = [out[len(inp):] for inp, out in zip(inputs.input_ids, outputs)]
    response = processor.batch_decode(generated, skip_special_tokens=True)[0]
    
    return extract_answer(response)

# Run inference on all test cases
print("Running inference on all test cases...\n")

all_predicted_time = []  # (id, time_in_ms)
all_result = []          # (id, answer)

for sample in tqdm(test_cases, desc="Processing"):
    sample_id = sample['id']
    question = sample['question']
    choices = sample['choices']
    video_path = Path("/data") / sample['video_path']
    
    # Skip if video not found
    if not video_path.exists():
        all_predicted_time.append((sample_id, 0))
        all_result.append((sample_id, "A"))
        continue
    
    # Measure time for this test case
    t1 = time.time()
    try:
        answer = predict_single(video_path, question, choices)
    except Exception as e:
        print(f"Error processing {sample_id}: {e}")
        answer = "A"
    t2 = time.time()
    
    predicted_time = int((t2 - t1) * 1000)  # Convert to milliseconds
    
    all_predicted_time.append((sample_id, predicted_time))
    all_result.append((sample_id, answer))

print(f"\n✅ Processed {len(test_cases)} test cases")

### Step 5: Save Results

In [ ]:
import pandas as pd

# Create output directory
os.makedirs("/result", exist_ok=True)

# Save submission results (jupyter_submission.csv)
df_submission = pd.DataFrame(all_result, columns=['id', 'answer'])
df_submission.to_csv('/result/jupyter_submission.csv', index=False)
print("✅ Saved jupyter_submission.csv")

# Save timing results (time_submission.csv)
df_time = pd.DataFrame(all_predicted_time, columns=['id', 'time'])
# Add answer column to time_submission.csv
df_time = df_time.merge(df_submission, on='id')
df_time.to_csv('/result/time_submission.csv', index=False)
print("✅ Saved time_submission.csv")

# Print summary
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"Total test cases: {len(test_cases)}")
print(f"Total time: {df_time['time'].sum()/1000:.2f}s ({df_time['time'].sum()} ms)")
print(f"Average time per case: {df_time['time'].mean():.0f} ms")
print(f"Min time: {df_time['time'].min()} ms")
print(f"Max time: {df_time['time'].max()} ms")

# Answer distribution
print(f"\nAnswer distribution:")
answer_counts = df_submission['answer'].value_counts().sort_index()
for ans, count in answer_counts.items():
    pct = 100 * count / len(test_cases)
    print(f"  {ans}: {count:4d} ({pct:5.1f}%)")

# Show first few results
print(f"\nFirst 5 results:")
print(df_time.head())

print(f"\n{'='*60}")
print("✅ Inference completed successfully!")
print(f"{'='*60}")